#Complex Types

####Arrays

In [0]:
# DBTITLE 1,Create and Display DataFrame with Complex Types
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, IntegerType
from pyspark.sql import functions as F

# Sample data
rows = [
    Row(
        id=1,
        names=["Alice", "Bob"],
        scores=[85, 90],
        info=Row(age=25, city="New York"),
        json_str='{"dept": "Engineering", "level": 2}'
    ),
    Row(
        id=2,
        names=["Carol"],
        scores=[78, 88, 92],
        info=Row(age=30, city="San Francisco"),
        json_str='{"dept": "HR", "level": 1}'
    )
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True),
    StructField("json_str", StringType(), True)
])

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("df")
display(df)

In [0]:
#Import code
from pyspark.sql import functions as F

# Get the length of the 'names' array
array_length_df = df.withColumn("names_length", F.size(F.col("names")))

# Check if 'names' array contains 'Alice'
array_contains_df = array_length_df.withColumn("has_Alice", F.array_contains(F.col("names"), "Alice"))

# Split a string column into an array (for demonstration, create a new column)
df_array = array_contains_df.withColumn("city_words", F.split(F.col("info.city"), " "))

#Display the data frame
display(df_array.select("id", "names", "names_length", "has_Alice", "city_words"))

In [0]:
%sql
SELECT
  id,
  names,
  size(names) AS names_length,
  array_contains(names, 'Alice') AS has_Alice,
  split(info.city, ' ') AS city_words
FROM df

####Exploding Arrays

In [0]:
#Brings all of the array elements into individual rows
exploded_df = df.select("id", F.explode(F.col("names")).alias("name"))

#Displays the data frame
display(exploded_df)

In [0]:
%sql
SELECT id, explode(names) AS name FROM df

####Struct Type

In [0]:
#Import code
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType

#Accessing fields in the info struct
struct_df = df.withColumn("age", F.col("info.age")).withColumn("city", F.col("info.city"))

#Display the data frame
display(struct_df)

In [0]:
%sql
SELECT *, info.age AS age, info.city AS city
  FROM df

####JSON

In [0]:
#Import code
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

#Extract a value using get_json_object
json_df = df.withColumn("dept", F.get_json_object(F.col("json_str"), "$.dept")) \
            .withColumn("level", F.get_json_object(F.col("json_str"), "$.level"))

#Display the data frame
display(json_df)

In [0]:
%sql
SELECT *,
  get_json_object(json_str, '$.dept') AS dept,
  get_json_object(json_str, '$.level') AS level
FROM df

####JSON to Struct

In [0]:
#Import code
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

#Parse JSON string into struct
json_schema = StructType([
    StructField("dept", StringType(), True),
    StructField("level", IntegerType(), True)
])
from_json_df = df.withColumn("json_struct", F.from_json(F.col("json_str"), json_schema))

#Accessing fileds in the info struct
struct_df = from_json_df.withColumn("dept", F.col("json_struct.dept"))

#Display the data frame
display(struct_df)

In [0]:
%sql
SELECT *,
  from_json(json_str, 'STRUCT<dept:STRING,level:INT>') AS json_struct,
  from_json(json_str, 'STRUCT<dept:STRING,level:INT>').dept AS dept
FROM df